# 04 — Your First RAG Pipeline

*Level 1 — Naive RAG*

## Objective
Assemble every previous notebook into the full loop: **load -> chunk -> embed -> store -> retrieve -> prompt -> LLM -> answer.**

**Requirements:** `ollama serve` running, plus `ollama pull nomic-embed-text` and `ollama pull llama3.2`.


In [1]:
import sys
from pathlib import Path

# Make the local `src` package importable from notebooks/
sys.path.insert(0, str(Path.cwd().parent))


In [2]:
from src.ingest import load_from_directory
from src.pipeline import RAGPipeline
from src.config import settings

# The small, hand-written docs in data/sample_docs/ — no download needed.
documents = load_from_directory(settings.sample_docs_dir)
print(f"Loaded {len(documents)} documents:")
for d in documents:
    print(f"  - {d.id} ({len(d.text.split())} words)")


Loaded 3 documents:
  - onboarding_faq (216 words)
  - product_overview (194 words)
  - refund_policy (168 words)


In [3]:
pipeline = RAGPipeline()  # Ollama embeddings + Ollama generation + in-memory store
n_chunks = pipeline.build_index(documents)
print(f"Indexed {n_chunks} chunks.")


Embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding chunks: 100%|██████████| 1/1 [00:02<00:00,  2.05s/it]

Embedding chunks: 100%|██████████| 1/1 [00:02<00:00,  2.05s/it]

Indexed 3 chunks.


## Ask a question


In [4]:
answer = pipeline.ask("How many days do I have to request a refund?")

print("Q:", answer.question)
print("A:", answer.answer)
print()
print("Sources:")
for i, source in enumerate(answer.sources, start=1):
    print(f"  [{i}] score={source.score:.3f} doc={source.chunk.document_id}")
    print(f"      {source.chunk.text[:150]}...")


Q: How many days do I have to request a refund?
A: According to Source 1 ([Source 1]), customers may request a full refund within **30 days** of purchase for unused and original packaging products. For digital products (software licenses, downloadable content), the refund window is **14 days** from purchase, as long as the license key has not been activated on more than one device.

Sources:
  [1] score=0.780 doc=refund_policy
      # Refund Policy Customers may request a full refund within **30 days** of purchase, provided the product is unused and in its original packaging. Refu...
  [2] score=0.488 doc=onboarding_faq
      # New Employee Onboarding FAQ ## When do I get my laptop? IT ships your laptop to arrive on or before your start date. If it has not arrived by 9 AM o...
  [3] score=0.436 doc=product_overview
      # Product Overview: Acme Notes Acme Notes is a note-taking application for small teams, available on desktop (Windows, macOS, Linux) and mobile (iOS, ...


## See it fail: ask something the corpus doesn't cover

The system will still retrieve its Top-K "closest" chunks — because brute-force search always returns *something* — and a naive LLM call may try to answer anyway. This is exactly the hallucination risk called out in the main README's Common Failure Modes.


In [5]:
answer = pipeline.ask("What is the CEO's home address?")
print("Q:", answer.question)
print("A:", answer.answer)


Q: What is the CEO's home address?
A: I don't know based on the provided context. The context does not mention the CEO's home address or any other personal information about company executives.


## Swap the generator to see retrieval quality in isolation

`ExtractiveGenerator` skips the LLM and returns the top retrieved chunk verbatim — useful for judging retrieval without generation quality mixed in.


In [6]:
from src.generate import ExtractiveGenerator

pipeline.generator = ExtractiveGenerator()
answer = pipeline.ask("How many days do I have to request a refund?")
print(answer.answer)


[Source 1] # Refund Policy Customers may request a full refund within **30 days** of purchase, provided the product is unused and in its original packaging. Refund requests must be submitted through the support portal at support.example.com/refunds along with the original order number. Digital products (software licenses, downloadable content) are refundable within **14 days** of purchase, as long as the license key has not been activated on more than one device. Enterprise customers on an annual contract follow a separate process: refunds for unused months are prorated and must be approved by an account manager. Enterprise refund requests typically take 5-7 business days to process, compared to 2-3 business days for standard consumer refunds. Shipping costs are non-refundable except in cases where the product arrived damaged or defective, in which case the customer should contact support within 48 hours of delivery with photos of the damage. Refunds are issued to the original payment 

## Next

- Try the full open-source dataset: `from src.ingest import load_from_hf_dataset` then `load_from_hf_dataset(limit=500)` — this downloads and caches `rag-datasets/rag-mini-wikipedia` the first time you run it (see [`../README.md#dataset`](../README.md#dataset)).
- Build the mini project: a local PDF question-answering assistant (swap `load_from_directory` to point at a folder of your own PDFs).
- Move on to [`02-advanced-rag`](../../02-advanced-rag/README.md) once you can explain *why* the second question above failed.
